# Task definition
The purpose of the project is to create some system to codify ICD-10 codes automatically. There are many different methods and approaches, so feel free to try any idea to make it work.

You will have to create a multiclass classifier that is able to codify diagnosis literals found in medical records into their respective ICD-10 category.

As a reference and to understand ICD codes, see the table below:

| Code Range  | Category                                                                                            |
| ----------- | --------------------------------------------------------------------------------------------------- |
| **A–B**     | Certain infectious and parasitic diseases                                                           |
| **C–D48**   | Neoplasms (cancers and tumors)                                                                      |
| **D50–D89** | Diseases of the blood and blood-forming organs and certain disorders involving the immune mechanism |
| **E**       | Endocrine, nutritional and metabolic diseases                                                       |
| **F**       | Mental and behavioural disorders                                                                    |
| **G**       | Diseases of the nervous system                                                                      |
| **H00–H59** | Diseases of the eye and adnexa                                                                      |
| **H60–H95** | Diseases of the ear and mastoid process                                                             |
| **I**       | Diseases of the circulatory system                                                                  |
| **J**       | Diseases of the respiratory system                                                                  |
| **K**       | Diseases of the digestive system                                                                    |
| **L**       | Diseases of the skin and subcutaneous tissue                                                        |
| **M**       | Diseases of the musculoskeletal system and connective tissue                                        |
| **N**       | Diseases of the genitourinary system                                                                |
| **O**       | Pregnancy, childbirth and the puerperium                                                            |
| **P**       | Certain conditions originating in the perinatal period                                              |
| **Q**       | Congenital malformations, deformations and chromosomal abnormalities                                |
| **R**       | Symptoms, signs and abnormal clinical and laboratory findings, not elsewhere classified             |
| **S–T**     | Injury, poisoning and certain other consequences of external causes                                 |
| **V–Y**     | External causes of morbidity and mortality                                                          |
| **Z**       | Factors influencing health status and contact with health services                                  |
| **U**       | Codes for special purposes (e.g., emerging diseases like **COVID-19 codes)                          |

Your objective is to create a model that, given a literal, it can predict to which category that diagnosis belongs.


> This notebook displays two different approaches as example: a machine learning approach with a vectorizer and a linear regression model and a similarity distance implementation with more simple computations.

# Problem example

## Import dependencies

In [1]:
from unidecode import unidecode
import pandas as pd
import re

# basic text normalization function 
def normalize(text):
    text = unidecode(text.lower())
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return text

### Load data to predict

In [ ]:
# load the data with literals to predict and normalize the literals
pred_df = pd.read_csv("uab-asho-ai-codification/UAB_codification_pred.csv", index_col='id')
pred_df['normalized_literal'] = pred_df['Literal'].apply(lambda x: normalize(x)).str.lower()
pred_df.head()

,Literal,normalized_literal
id,,
1,DISTOCIA DE HOMBROS,distocia de hombros
2,Grupo 0+,grupo 0
3,Desgarro tipo I,desgarro tipo i
4,salpingooforectomía bilateral,salpingooforectomia bilateral
5,Quiste ovario derecho,quiste ovario derecho


### Load data to train/baseline

In [3]:
# load the data that will be used to train or to have as the baseline + normalization
target_df = pd.read_csv("uab-asho-ai-codification/UAB_icd_d_p.csv")
target_df['normalized_desc'] = target_df['Description'].apply(lambda x: normalize(x)).str.lower()
target_df.head()

,Code,D_P,Description,normalized_desc
0,A00,D,Cólera,colera
1,A000,D,"Cólera debido a Vibrio cholerae 01, biotipo ch...",colera debido a vibrio cholerae 01 biotipo ch...
2,A001,D,"Cólera debido a Vibrio cholerae 01, biotipo El...",colera debido a vibrio cholerae 01 biotipo el...
3,A009,D,"Cólera, no especificado",colera no especificado
4,A01,D,Fiebres tifoidea y paratifoidea,fiebres tifoidea y paratifoidea


In [4]:
# create target labels
from sklearn.preprocessing import LabelEncoder

# use label encoder to transform the categories into integers (classes)
le = LabelEncoder()

# get the first digit of the ICD codes and treat it like main category
# add a column with integer labels for each category
target_df['category'] = target_df['Code'].str[0]
target_df["category_id"] = le.fit_transform(target_df["category"])

# drop code column (not intended)
target_df = target_df.drop(columns=["Code"])

target_df.head()

,D_P,Description,normalized_desc,category,category_id
0,D,Cólera,colera,A,10
1,D,"Cólera debido a Vibrio cholerae 01, biotipo ch...",colera debido a vibrio cholerae 01 biotipo ch...,A,10
2,D,"Cólera debido a Vibrio cholerae 01, biotipo El...",colera debido a vibrio cholerae 01 biotipo el...,A,10
3,D,"Cólera, no especificado",colera no especificado,A,10
4,D,Fiebres tifoidea y paratifoidea,fiebres tifoidea y paratifoidea,A,10


### Load ground truth for prediction

In [5]:
# load the literals with its correct codes (ground truth results)
ground_truth = pd.read_csv("UAB_codification_ground_truth.csv").drop(columns=['Usage'])

# set categories and set categories ids
ground_truth['category'] = ground_truth['Code'].str[0]
ground_truth['category_id'] = le.fit_transform(ground_truth['category'])

# remove 'Code' column since its not necessary
ground_truth = ground_truth.drop(columns=['Code'])

ground_truth.head()

,id,Literal,category,category_id
0,1,DISTOCIA DE HOMBROS,O,24
1,2,Grupo 0+,Z,35
2,3,Desgarro tipo I,O,24
3,4,salpingooforectomía bilateral,0,0
4,5,Quiste ovario derecho,N,23


## Codification with similarity

When codifying with similarity we will assume that each code will have a single expression or literals associated (which is not true, since there are many ways to name a same diagnosis).

Idea: all literals to predict will be vectorized along with all entities in the full list. The distance between vectors will be computed and the literals with minimal distance will return the expected correct code.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# create a vectorizer
vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3,5)
)

# set the vectorizer with the descriptions of the target data and transform the literals to predict
X_icd = vectorizer.fit_transform(target_df['normalized_desc'])
X_lit = vectorizer.transform(pred_df['normalized_literal'])

### Compute similarities and retrieve most similar literal

Notice that with this approach we will get a full code, since the returned code is taken from the original full list.

In [ ]:
# compute cosine similarity between the literals and the descriptions
sim = cosine_similarity(X_lit, X_icd)
best = sim.argmax(axis=1)

pred_codes = target_df.iloc[best]["category"].values

pred_df['predicted_category'] = pred_codes
pred_df.head()

,Literal,normalized_literal,predicted_category
id,,,
1,DISTOCIA DE HOMBROS,distocia de hombros,O
2,Grupo 0+,grupo 0,Z
3,Desgarro tipo I,desgarro tipo i,S
4,salpingooforectomía bilateral,salpingooforectomia bilateral,O
5,Quiste ovario derecho,quiste ovario derecho,N


### Compute metrics

In [ ]:
from sklearn.metrics import classification_report

# compute accuracys
accuracy = (ground_truth['category'].values == pred_df["predicted_category"].values).mean()
print("Accuracy:", accuracy, "\n")

print(classification_report(ground_truth['category'], pred_df["predicted_category"], zero_division=0))

Accuracy: 0.3472398880691936 

              precision    recall  f1-score   support

           0       0.33      0.09      0.14       334
           1       0.00      0.00      0.00       110
           2       0.00      0.00      0.00        64
           3       0.25      0.01      0.02       232
           4       0.21      0.05      0.08        86
           5       0.00      0.00      0.00        60
           6       1.00      0.02      0.03       122
           7       0.00      0.00      0.00        66
           8       1.00      0.01      0.02       116
           9       0.00      0.00      0.00        57
           A       0.03      0.75      0.05         4
           B       0.62      0.46      0.53       159
           C       0.19      0.20      0.20        45
           D       0.33      0.81      0.47        88
           E       0.43      0.64      0.51       163
           F       0.35      0.44      0.39        88
           G       0.44      0.82      0.58       

> More text processing may be needed for an improvement in results, but as a baseline it is an interesting approach!

## Codification with Machine Learning

Codification with ML needs training and labeled data but can give better results. We will be using a vectorizer, as before, but this time we will also train a logistic regressor to learn each category distribution and classify between categories. This approach will only predict the main category (first digit in ICD code), but it can be a nice beginning for a hierarchical implementation.

> Notice that we are using the full list data rather than the data gathered from the original reports, so domain is not exclusively aligned. There are other ways to train the model.

### Prepare training

In [12]:
# create training variables (X and y)
X_train = target_df['normalized_desc']
y_train = target_df['category_id']

In [13]:
# import model from scikit-learn 
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# create model pipeline
model = Pipeline([
    ("features", FeatureUnion([
        ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5))),
        ("word", TfidfVectorizer(analyzer="word", ngram_range=(1,2)))
    ])),
    ("clf", LogisticRegression(max_iter=5000))
])

# train model
model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('features', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformer_list transformer_list: list of (str, transformer) tuplesList of transformer objects to be applied to the data. The firsthalf of each tuple is the name of the transformer. The transformer canbe 'drop' for it to be ignored or can be 'passthrough' for features tobe passed unchanged... versionadded:: 1.1 Added the option `""passthrough""`... versionchanged:: 0.22 Deprecated `None` as a transformer in favor of 'drop'.","[('char', ...), ('word', ...)]"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer.Keys are transformer names, values the weights.Raises ValueError if key not present in ``transformer_list``.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, default=TrueIf True, :meth:`get_feature_names_out` will prefix all feature nameswith the name of the transformer that generated that feature.If False, :meth:`get_feature_names_out` will not prefix any featurenames and will error if feature names are not unique... versionadded:: 1.5",True
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'


### Compute inference on prediction data

In [14]:
# predict with trained model
pred_df["predicted_category_id"] = model.predict(pred_df["normalized_literal"])
# convert the labels to the original category (first digit from code)
pred_df["predicted_category"] = le.inverse_transform(pred_df['predicted_category_id'])

pred_df.head()

,Literal,normalized_literal,predicted_category,predicted_category_id
id,,,,
1,DISTOCIA DE HOMBROS,distocia de hombros,M,22
2,Grupo 0+,grupo 0,X,33
3,Desgarro tipo I,desgarro tipo i,S,28
4,salpingooforectomía bilateral,salpingooforectomia bilateral,H,17
5,Quiste ovario derecho,quiste ovario derecho,N,23


### Compute metrics

In [ ]:
# compute accuracy
accuracy = (ground_truth["category"].values == pred_df["predicted_category"].values).mean()
print("Accuracy:", accuracy, '\n')

print(classification_report(ground_truth["category"], pred_df["predicted_category"], zero_division=0))

Accuracy: 0.3195115746629356 

              precision    recall  f1-score   support

           0       0.08      0.00      0.01       334
           1       0.00      0.00      0.00       110
           2       0.00      0.00      0.00        64
           3       1.00      0.01      0.03       232
           4       1.00      0.06      0.11        86
           5       0.00      0.00      0.00        60
           6       0.00      0.00      0.00       122
           7       0.00      0.00      0.00        66
           8       0.00      0.00      0.00       116
           9       0.00      0.00      0.00        57
           A       0.02      0.25      0.03         4
           B       0.57      0.53      0.55       159
           C       0.32      0.18      0.23        45
           D       0.24      0.69      0.36        88
           E       0.50      0.67      0.57       163
           F       0.58      0.35      0.44        88
           G       0.42      0.68      0.52       

> These results are slightly better than the similarity implementation, but can also be improved by changing the data used for training, the models or the text processing. Feel free to experiment and achieve better results! 